# Dilbert

Dilbert is the interactive Zenoh tool for Nachtlicht. Execute the cells from top to bottom. The notebook discovers a local Zenoh router, requests colour changes and displays the ESP32 events.

## 1. Define the Zenoh keys

Dilbert discovers a local Zenoh router in the next cell. No router address needs to be configured.

In [1]:
COLOR_NEXT_KEYEXPR = "nachtlicht/led/color/next"
COLOR_CHANGED_KEYEXPR = "nachtlicht/led/color/changed"
BRIGHTNESS_CHANGED_KEYEXPR = "nachtlicht/led/brightness/changed"
COLOR_NEXT_PAYLOAD = "next"

print(f"Command key: {COLOR_NEXT_KEYEXPR}")
print(f"Event keys: {COLOR_CHANGED_KEYEXPR}, {BRIGHTNESS_CHANGED_KEYEXPR}")

Command key: nachtlicht/led/color/next
Event keys: nachtlicht/led/color/changed, nachtlicht/led/brightness/changed


## 2. Open a Zenoh session

Zenoh scouts for a local router via UDP multicast at `224.0.0.224:7446`. The router announces the TCP locator used to open the session.

In [2]:
import json

import zenoh

config = zenoh.Config()
config.insert_json5("mode", json.dumps("client"))
session = zenoh.open(config)
print("Zenoh session opened after scouting.")

Zenoh session opened after scouting.


## 3. Declare the publisher and event subscribers

The publisher sends colour-change requests. The subscribers receive the ESP32 colour and brightness events through thread-safe Zenoh queues; the GUI consumes those queues in the Jupyter event loop.

In [3]:
publisher = session.declare_publisher(COLOR_NEXT_KEYEXPR)
color_changed_subscriber = session.declare_subscriber(COLOR_CHANGED_KEYEXPR)
brightness_changed_subscriber = session.declare_subscriber(BRIGHTNESS_CHANGED_KEYEXPR)
print(f"Publisher declared for {COLOR_NEXT_KEYEXPR}.")
print("Event subscribers declared.")

Publisher declared for nachtlicht/led/color/next.
Event subscribers declared.


## 4. Control and observe Nachtlicht

Click the button to request the next colour. The asynchronous task polls the Zenoh subscriber queues without blocking the notebook, then updates the widgets from the Jupyter event loop. The brightness-event counter is the live heartbeat.

In [4]:
import asyncio
import json

import ipywidgets as widgets
from IPython.display import display

color_events = 0
brightness_events = 0

next_color_button = widgets.Button(description="Next colour", button_style="primary")
color_status = widgets.HTML(value="Last colour: waiting for an event")
brightness_status = widgets.HTML(value="Brightness heartbeat: waiting for an event")

def request_next_colour(_button):
    publisher.put(COLOR_NEXT_PAYLOAD)

def update_colour(sample):
    global color_events
    event = json.loads(sample.payload.to_string())
    color_events += 1
    color_status.value = f"Last colour: {event['color']} (event {event['sequence']}, received {color_events})"

def update_brightness(sample):
    global brightness_events
    event = json.loads(sample.payload.to_string())
    brightness_events += 1
    brightness_status.value = f"Brightness heartbeat: {event['brightness']} (event {event['sequence']}, received {brightness_events})"

async def receive_events():
    try:
        while True:
            while sample := color_changed_subscriber.try_recv():
                update_colour(sample)
            while sample := brightness_changed_subscriber.try_recv():
                update_brightness(sample)
            await asyncio.sleep(0.1)
    except (asyncio.CancelledError, zenoh.ZError):
        return

next_color_button.on_click(request_next_colour)
event_task = asyncio.create_task(receive_events())
display(widgets.VBox([next_color_button, color_status, brightness_status]))

## 5. Close the session

Run this cell before restarting the kernel or shutting down JupyterLab. It stops the event task before closing the Zenoh resources.

In [ ]:
event_task.cancel()
try:
    await event_task
except asyncio.CancelledError:
    pass
color_changed_subscriber.undeclare()
brightness_changed_subscriber.undeclare()
publisher.undeclare()
session.close()
print("Zenoh session closed.")